In [ ]:
INTENT_CLASSIFICATION_PROMPT = """
Phân tích câu hỏi của người dùng và phân loại ý định vào một trong các nhãn sau: [TRA_CUU, SO_SANH, PHAN_TICH, KHAC].
Đồng thời, trích xuất các thực thể liên quan. Trả về kết quả dưới dạng JSON.

# Yêu cầu phân tích:
1. Phân loại intent của câu hỏi người dùng vào **một trong bốn nhãn sau**:
    - "TRA_CUU": Câu hỏi mang tính chất tra cứu khái niệm, định nghĩa hoặc quy định cụ thể.
    - "SO_SANH": Câu hỏi yêu cầu so sánh giữa hai hay nhiều luật, văn bản hoặc giai đoạn pháp lý khác nhau.
    - "PHAN_TICH": Câu hỏi yêu cầu phân tích, giải thích, lập luận hoặc đánh giá một vấn đề pháp lý.
    - "KHAC": Câu hỏi không thuộc các loại trên, yêu cầu ngoài phạm vi pháp lý rõ ràng.

2. Xác định "topic": Chủ đề chính của câu hỏi, không bao gồm phần entities, tìm và loại bỏ tất cả các từ liên quan đến intent . (ví dụ: "phân tích", "sự khác biệt", "so sánh", "giải thích",...). Nếu có các thành phần nhỏ như chương, điều, khoản hoặc các mục I,II,1,1.1,... thì thêm vào topic.

3. Trích xuất "entities": Danh sách các thực thể pháp lý cụ thể xuất hiện trong câu hỏi, như: tên luật, năm ban hành, nghị định, thông tư, văn bản, cơ quan hoặc tổ chức cụ thể. Nếu là các thành phần nhỏ như chương, điều, khoản hoặc các mục I,II,1,1.1,... thì không thêm vào entities mà giữ lại ở phần topic.

# Yêu cầu bắt buộc:
- Chỉ trả về kết quả dưới dạng JSON, không trả lời thêm gì khác.
- Không trả về các chi tiết thừa như "```json ```"

Các ví dụ:
- Câu hỏi: "Quy phạm pháp luật là gì?" -> {"intent": "TRA_CUU", "topic": "quy phạm pháp luật", "entities": []}
- Câu hỏi: "Phân tích các trường hợp được đơn phương chấm dứt hợp đồng." -> {"intent": "PHAN_TICH", "topic": "các trường hợp được đơn phương chấm dứt hợp đồng", "entities": []}
- Câu hỏi: "So sánh Luật Doanh nghiệp 2014 và 2020 về vốn điều lệ" -> {"intent": "SO_SANH", "topic": "vốn điều lệ", "entities": ["Luật Doanh nghiệp 2014", "Luật Doanh nghiệp 2020"]}
- Câu hỏi: "So sánh giữa Luật Hôn nhân và Gia đình 2000, 2014 và Bộ luật Dân sự 2015 về quyền nuôi con" -> {"intent": "SO_SANH", "topic": "quyền nuôi con", "entities": ["Luật Hôn nhân và Gia đình 2000", "Luật Hôn nhân và Gia đình 2014", "Bộ luật Dân sự 2015"]}
- Câu hỏi: "Bạn có thể phân tích nội dung chính của Nghị quyết 68/NQ-CP không?" -> {"intent": "PHAN_TICH", "topic": "nội dung chính", "entities": ["Nghị quyết 68/NQ-CP"]}
- Câu hỏi: "So sánh sự khác biệt giữa điều 5 chương 3 của Bộ luật Hình sự năm 2015 và 2019" -> {"intent": "SO_SANH", "topic": "điều 5 chương 3", "entities": ["Bộ luật Hình sự năm 2015", "Bộ luật Hình sự năm 2019"]}

Câu hỏi cần phân tích: "{user_query}"
"""

In [ ]:
EXPANSION_PROMPT = """
Bạn là một chuyên gia pháp luật với nhiều năm kinh nghiệm phân tích chuyên sâu các quy định pháp lý của Việt Nam.
Nhiệm vụ của bạn là **phân tích chuyên sâu một chủ đề pháp lý cụ thể** do người dùng cung cấp, bằng cách mở rộng và chia nhỏ chủ đề thành **các khía cạnh pháp lý quan trọng và có liên quan nhất** để phục vụ cho mục đích truy xuất mở rộng thông tin pháp luật.
Đồng thời, trả về kết quả dưới dạng JSON.

# Yêu cầu:

1. **Hiểu chủ đề đầu vào**:
   - Phân tích ý nghĩa pháp lý của chủ đề.
   - Hiểu cách chủ đề đó thường được đề cập hoặc xử lý trong các văn bản quy phạm pháp luật Việt Nam.
   - Xem xét các bối cảnh thường gặp trong thực tiễn pháp lý có liên quan đến chủ đề này.

2. **Mở rộng thành các khía cạnh cụ thể**:
   - Trích xuất và liệt kê các **khía cạnh quan trọng**, thường là các điểm pháp lý khác nhau cần làm rõ khi xử lý chủ đề đó.
   - Mỗi khía cạnh nên ngắn gọn, rõ nghĩa, có thể sử dụng như một câu truy vấn độc lập trong hệ thống.
   - Mỗi khía cạnh nên đại diện cho một góc nhìn hoặc phạm vi nội dung khác nhau liên quan đến chủ đề (ví dụ: quy định, điều kiện áp dụng, thủ tục, thời hạn, trách nhiệm, xử lý vi phạm, ngoại lệ...).

# Yêu cầu bắt buộc:
- Chỉ trả về kết quả dưới dạng JSON, không trả lời thêm gì khác.
- Không trả về các chi tiết thừa như "```json ```"
- Trả về đúng 4 khía cạnh mở rộng của chủ đề.
- Các khía cạnh mở rộng phải tối ưu hóa tối đa để sử dụng làm truy vấn cho hệ thống RAG sử dụng **similarity search**.


# Ví dụ:
**Chủ đề**: "chấm dứt hợp đồng lao động"

**Các khía cạnh phân tích**:
1. Các trường hợp được phép chấm dứt hợp đồng lao động theo quy định pháp luật
2. Trình tự, thủ tục chấm dứt hợp đồng lao động hợp pháp
3. Trách nhiệm của người sử dụng lao động khi chấm dứt hợp đồng
4. Quyền lợi người lao động khi bị chấm dứt hợp đồng trái pháp luật

**Kết quả trả về**:
{"topic": "chấm dứt hợp đồng lao động", "sub_topic":[
    "Các trường hợp được phép chấm dứt hợp đồng lao động theo quy định pháp luật",
    "Trình tự, thủ tục chấm dứt hợp đồng lao động hợp pháp",
    "Trách nhiệm của người sử dụng lao động khi chấm dứt hợp đồng",
    "Quyền lợi người lao động khi bị chấm dứt hợp đồng trái pháp luật"
]}

Chủ đề cần phân tích: "{topic}"
"""

In [ ]:
!pip install -U langchain-openai langchain-chroma langchain-community langchain-huggingface --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 5.3 MB/s eta 0:00:00


In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.docstore.document import Document
from langchain.text_splitter import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="AITeamVN/Vietnamese_Embedding",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")

openai_api_key = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(
    model = 'gpt-4o',
    openai_api_key = openai_api_key,
    temperature=0
)

/tmp/ipython-input-10-1942192328.py:8: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(


In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.schema import HumanMessage
import json


def get_question_json(user_query: str) -> dict:

    prompt = INTENT_CLASSIFICATION_PROMPT.replace("{user_query}", user_query)

    response = llm([HumanMessage(content=prompt)])

    try:
        result = json.loads(response.content)
    except json.JSONDecodeError:
        raise ValueError(f"Lỗi khi parse JSON từ output của LLM:\n{response.content}")

    return result

In [ ]:
def get_expansion_topic(topic: str) -> dict:

    prompt = EXPANSION_PROMPT.replace("{topic}", topic)

    response = llm([HumanMessage(content=prompt)])

    try:
        result = json.loads(response.content)
    except json.JSONDecodeError:
        raise ValueError(f"Lỗi khi parse JSON từ output của LLM:\n{response.content}")

    return result

In [ ]:
question1 = "So sánh sự khác biệt giữa mục 3 chương 8 của luật lao động năm 2014 và 2022"
question2 = "Hợp đồng lao động là gì?"
question3 = "Giải thích các trường hợp được quy định trong chương 3 luật hình sự năm 2015"
question4 = "Phân tích thủ tục cấp giấy phép xây dựng nhà ở riêng lẻ trong luật đất đai năm 2016"

In [ ]:
question_json = get_question_json(question4)
expansion_topic_json = get_expansion_topic_json(question_json['topic'])
print(question_json)
print(expansion_topic_json)

{'intent': 'PHAN_TICH', 'topic': 'thủ tục cấp giấy phép xây dựng nhà ở riêng lẻ', 'entities': []}
{'topic': 'thủ tục cấp giấy phép xây dựng nhà ở riêng lẻ', 'sub_topic': ['Điều kiện cần thiết để được cấp giấy phép xây dựng nhà ở riêng lẻ.', 'Trình tự và thủ tục nộp hồ sơ xin cấp giấy phép xây dựng nhà ở riêng lẻ.', 'Thời hạn giải quyết hồ sơ xin cấp giấy phép xây dựng nhà ở riêng lẻ.', 'Trách nhiệm và nghĩa vụ của chủ đầu tư sau khi được cấp giấy phép xây dựng nhà ở riêng lẻ.']}


In [ ]:
# @title Văn bản tiêu đề mặc định
query = []

for i in range(len(result['entities'])):
    sub_query = ''
    sub_query += result['topic'] + ' ' + result['entities'][i]
    query.append(sub_query)

print(query)

NameError: name 'result' is not defined

In [ ]:
def get_sub_query(topic, entities, query):
    if len(entities) == 0:
        query.append(topic)
        return
    for i in range(len(entities)):
        sub_query = ''
        sub_query += topic + ' ' + entities[i]
        query.append(sub_query)

In [ ]:
topic_test = result['topic']
entities_test = result['entities']
list_query = []

get_sub_query(topic_test, entities_test, list_query)
print(list_query)

NameError: name 'result' is not defined

In [ ]:
def format_question(intent, topic, entities):
    intent_map = {
        "TRA_CUU": "Tra cứu",
        "SO_SANH": "So sánh",
        "PHAN_TICH": "Phân tích",
        "KHAC": ""
    }
    intent_formated = intent_map.get(intent, intent)

    if not entities:
        question = f"{intent_formated} {topic}"
    elif len(entities) == 1:
        question = f"{intent_formated} {topic} {entities[0]}"
    else:
        entity_str = ' và '.join(entities)
        question = f"{intent_formated} {topic} {entity_str}"

    return question

In [ ]:
question_formated = format_question(result['intent'], result['topic'], result['entities'])
print(question_formated)

NameError: name 'result' is not defined

In [ ]:
# Extract queries

def extract_query(question_json, topic_json = None):
    intent = question_json['intent']
    topic = question_json['topic']
    entities = question_json['entities']
    sub_topics = topic_json['sub_topic'] if topic_json else []
    query = []

    def get_sub_query(topic, entities):
        if not entities:
            query.append(topic)
        else:
            for entity in entities:
                sub_query = f"{topic} {entity}"
                query.append(sub_query)


    def get_expansion_sub_query(sub_topics, entities):
        if not sub_topics:
             if not entities:
                query.append(topic)
             else:
                for entity in entities:
                    sub_query = f"{topic} {entity}"
                    query.append(sub_query)
        else:
            if not entities:
                for sub_topic in sub_topics:
                    query.append(sub_topic)
            else:
                for entity in entities:
                    for sub_topic in sub_topics:
                        sub_query = f"{sub_topic} {entity}"
                        query.append(sub_query)


    match intent:
        case 'TRA_CUU' | 'SO_SANH' | 'KHAC':
            get_sub_query(topic, entities)
        case 'PHAN_TICH':
            get_expansion_sub_query(sub_topics, entities)

    return query

In [ ]:
def format_question(question_json):
    intent = question_json['intent']
    topic = question_json['topic']
    entities = question_json['entities']
    intent_map = {
        "TRA_CUU": "Tra cứu",
        "SO_SANH": "So sánh",
        "PHAN_TICH": "Phân tích",
        "KHAC": ""
    }
    intent_formated = intent_map.get(intent, intent)

    if not entities:
        question = f"{intent_formated} {topic}"
    elif len(entities) == 1:
        question = f"{intent_formated} {topic} {entities[0]}"
    else:
        entity_str = ' và '.join(entities)
        question = f"{intent_formated} {topic} {entity_str}"

    return question

In [ ]:
def extract_expansion_topic_query(topic_json):

    topic = topic_json.get("topic", "")
    sub_queries = topic_json.get("sub_topic", [])

    return topic, sub_queries

In [ ]:
question1 = "Phân tích sự khác biệt trong cách tổ chức thực hiện giữa Nghị quyết 1672/NQ-UBTVQH15 (Lạng Sơn) và Nghị quyết 1671/NQ-UBTVQH15 (Lâm Đồng), đặc biệt về trách nhiệm của Chính phủ trong việc xác định ranh giới và công bố diện tích tự nhiên."

In [ ]:
question_json = get_question_json(question1)
expansion_topic_json = get_expansion_topic(question_json['topic'])
question_query = extract_query(question_json, expansion_topic_json)
formated_question = format_question(question_json)
expansion_topic, expansion_query = extract_expansion_topic_query(expansion_topic_json)
print(question_json)
print(question_query)
print(formated_question)
print(expansion_topic)
print(expansion_query)

{'intent': 'PHAN_TICH', 'topic': 'sự khác biệt trong cách tổ chức thực hiện, đặc biệt về trách nhiệm của Chính phủ trong việc xác định ranh giới và công bố diện tích tự nhiên', 'entities': ['Nghị quyết 1672/NQ-UBTVQH15 (Lạng Sơn)', 'Nghị quyết 1671/NQ-UBTVQH15 (Lâm Đồng)']}
['Quy định pháp luật về trách nhiệm của Chính phủ trong xác định ranh giới tự nhiên Nghị quyết 1672/NQ-UBTVQH15 (Lạng Sơn)', 'Thủ tục và quy trình công bố diện tích tự nhiên theo quy định hiện hành Nghị quyết 1672/NQ-UBTVQH15 (Lạng Sơn)', 'Sự khác biệt trong tổ chức thực hiện giữa các cấp chính quyền về xác định ranh giới Nghị quyết 1672/NQ-UBTVQH15 (Lạng Sơn)', 'Trách nhiệm và quyền hạn của các cơ quan liên quan trong công bố diện tích tự nhiên Nghị quyết 1672/NQ-UBTVQH15 (Lạng Sơn)', 'Quy định pháp luật về trách nhiệm của Chính phủ trong xác định ranh giới tự nhiên Nghị quyết 1671/NQ-UBTVQH15 (Lâm Đồng)', 'Thủ tục và quy trình công bố diện tích tự nhiên theo quy định hiện hành Nghị quyết 1671/NQ-UBTVQH15 (Lâm Đồng